Notebook to orchestrate ingesting data from the CitiBike S3 https://s3.amazonaws.com/tripdata/index.html. Imports functions from src.bronze.download_utils

In [0]:
catalog = "citibike"
schema = "landing"
volume = "raw"
volume_path = f"/Volumes/{catalog}/{schema}/{volume}"

# Run once to make sure volume exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

Build the list of downloads. For 2014-2023, there is one zip file per year. For 202401-202605, there is one zip file per month.

In [0]:
downloads = []

base_url = "https://s3.amazonaws.com/tripdata"
local_tmp = "/tmp/citibike"

for year in range(2014, 2024):
    downloads.append({
        "url": f"{base_url}/{year}-citibike-tripdata.zip",
        "zip_name": f"{year}-citibike-tripdata.zip",
        "era": "yearly"
    })

# create a list like: ['202401', '202402', '202403', ...]
year_months = []
for year in [2024, 2025, 2026]:
    for month in range(1, 13):
        ym = f"{year}{month:02d}"
        if ym > "202605":
            break
        if ym < "202401":
            continue
        year_months.append(ym)

for ym in year_months:
    downloads.append({
        "url": f"{base_url}/{ym}-citibike-tripdata.zip",
        "zip_name": f"{ym}-citibike-tripdata.zip",
        "era": "monthly"
    })

print(f"{len(downloads)} files")

In [0]:
downloads

Loop that downloads and extracts into temporary directories, combines data, and loads to volumes and delete temporary directories

In [0]:
import os
import shutil
import re

from src.bronze.download_utils import (
    run, resolve_month_key, combine_csvs, extract_yearly_zip, extract_monthly_zip
)

os.makedirs(local_tmp, exist_ok=True)

skipped = []
loaded = []

for item in downloads:
    url = item["url"]
    zip_name = item["zip_name"]
    era = item["era"]
    zip_path = f"{local_tmp}/{zip_name}"
    extract_dir = f"{local_tmp}/extract_{zip_name.replace('.zip','')}"
    combine_dir = f"{local_tmp}/combined_{zip_name.replace('.zip','')}"

    print(f"Processing {zip_name} ({era})...")

    rc, out, err = run(f"curl -sf '{url}' -o '{zip_path}'")

    # If not successful, add zip_name to skipped list
    if rc != 0:
        print(f"Skipped: download failed ({err.strip()[:200]})")
        skipped.append(zip_name)
        continue

    try:
        if era == "yearly":
            # get year from first 4 digits
            year = re.match(r"^(\d{4})", zip_name).group(1)
            
            # dictionary mapping month_keys to list of csv paths
            month_csvs = extract_yearly_zip(zip_path, extract_dir, year)

        # if era is monthly
        else:
            # get year month from first 6 digits
            ym = re.match(r"^(\d{6})", zip_name).group(1)

            # dictionary mapping month_keys to list of csv paths
            month_csvs = extract_monthly_zip(zip_path, extract_dir, ym)
    
    # If not successful, add zip_name to skipped list
    except Exception as e:
        print(f"Skipped: extraction failed ({e})")
        skipped.append(zip_name)
        continue

    # If month_csvs is empty
    if not month_csvs:
        print(f"Warning: no month data found in {zip_name}")
        skipped.append(zip_name)
        if os.path.exists(zip_path):
            os.remove(zip_path)
        if os.path.exists(extract_dir):
            shutil.rmtree(extract_dir)
        continue

    os.makedirs(combine_dir, exist_ok=True)

    for month_key, csv_paths in sorted(month_csvs.items()):
        month_folder_name = f"{month_key}-citibike-tripdata"
        combined_csv_name = f"{month_key}-citibike-tripdata.csv"
        local_combined_path = f"{combine_dir}/{combined_csv_name}"

        combined_csv = combine_csvs(csv_paths, local_combined_path)
        if not combined_csv:
            print(f"Warning: could not combine CSVs for {month_key}")
            continue

        year_key = month_key[:4]
        dest_dir = f"{volume_path}/{year_key}-citibike-tripdata/{month_folder_name}"

        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(local_combined_path, f"{dest_dir}/{combined_csv_name}")

        loaded.append(f"{year_key}-citibike-tripdata/{month_folder_name}/{combined_csv_name}")
        print(f"Loaded {month_folder_name} ({len(csv_paths)} source CSV(s) combined)")

    # Remove the temporary directories
    for path in (zip_path, extract_dir, combine_dir):
        if os.path.isfile(path):
            os.remove(path)
        elif os.path.isdir(path):
            shutil.rmtree(path)

print(f"\nDone. {len(loaded)} month CSVs loaded, {len(skipped)} zips skipped.")
if skipped:
    print("Skipped:", skipped)